In [1]:
!pip install streamlit pyngrok bcrypt PyJWT

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 52.3 MB/s eta 0:00:00


In [2]:
!pip install textstat py-cpuinfo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.6 MB/s eta 0:00:00


In [3]:
%%writefile app.py
import streamlit as st
import sqlite3
import re
import jwt
import datetime
import bcrypt
import base64
import os
import time
import random
import textstat
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- Configuration & Security ---
SECRET_KEY = "policy_nav_secret_key"
st.set_page_config(page_title="PolicyNav | Milestone 2", layout="wide")

# --- SMTP Email Configuration ---
# REPLACE THESE WITH YOUR ACTUAL CREDENTIALS FOR THE DEMO
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
SENDER_EMAIL = "mansichatur1504@gmail.com"
SENDER_PASSWORD = "zzdv pgcd ygwc lqdd"

# --- Database Initialization ---
def init_db():
    conn = sqlite3.connect("users.db", check_same_thread=False)
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS users(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT,
        email TEXT UNIQUE,
        password TEXT,
        security_question TEXT,
        security_answer TEXT
    )
    """)
    conn.commit()
    return conn, cursor

conn, cursor = init_db()

# --- Helper Functions ---
def send_otp_email(receiver_email, otp_code):
    subject = "PolicyNav Access Code"
    body = f"Hello,\n\nYour One-Time Password (OTP) for PolicyNav is: {otp_code}\n\nDo not share this code with anyone."
    msg = MIMEMultipart()
    msg['From'] = SENDER_EMAIL
    msg['To'] = receiver_email
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))
    try:
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.send_message(msg)
        server.quit()
        return True
    except Exception as e:
        st.error(f"Email Sending Failed: {e}")
        return False

def get_base64(file):
    if os.path.exists(file):
        with open(file, "rb") as f:
            return base64.b64encode(f.read()).decode()
    return ""

def password_strength(password):
    checks = {
        "Length ≥ 8": len(password) >= 8,
        "Uppercase": re.search(r"[A-Z]", password),
        "Lowercase": re.search(r"[a-z]", password),
        "Number": re.search(r"[0-9]", password),
        "Special": re.search(r"[!@#$%^&*(),.?\":{}|<>]", password)
    }
    return checks

def valid_password(password):
    return all(password_strength(password).values())

def valid_email(email):
    pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
    return re.match(pattern, email)

def hash_data(data):
    return bcrypt.hashpw(data.encode(), bcrypt.gensalt()).decode()

def check_data(data, hashed):
    try:
        return bcrypt.checkpw(data.encode(), hashed.encode())
    except:
        return False

# --- UI Styling ---
bg_base64 = get_base64("bg.png")
st.markdown(f"""
<style>
.stApp {{
    background-image: url("data:image/png;base64,{bg_base64}");
    background-size: cover;
}}
html, body, p, span, label, div {{ color: #0B3C5D !important; }}
h1, h2, h3, h4 {{ color: #0B3C5D !important; }}

div.stButton > button {{
    background-color: #FFFFFF !important;
    color: #0B3C5D !important;
    border: 2px solid #0B3C5D !important;
    font-weight: bold !important;
    border-radius: 5px !important;
}}
div.stButton > button:hover {{
    background-color: #0B3C5D !important;
    color: #FFFFFF !important;
}}

div[data-baseweb="select"]::after {{
    content: "▼";
    color: #0B3C5D;
    position: absolute;
    right: 15px;
    top: 12px;
    pointer-events: none;
}}
</style>
""", unsafe_allow_html=True)

# --- Session Management ---
if "logged_in" not in st.session_state:
    st.session_state.logged_in = False
if "otp_verified" not in st.session_state:
    st.session_state.otp_verified = False
if "generated_otp" not in st.session_state:
    st.session_state.generated_otp = None

# --- APP FLOW ---

# 1. READABILITY DASHBOARD
if st.session_state.logged_in and st.session_state.otp_verified:
    st.sidebar.title("PolicyNav")
    if st.sidebar.button("Logout"):
        st.session_state.clear()
        st.rerun()

    st.title("📜 Readability Dashboard")
    st.header(f"Welcome, {st.session_state.username}")

    policy_input = st.text_area("Analyze Public Policy Text", height=250, placeholder="Paste policy text here...")

    if policy_input:
        st.divider()
        c1, c2, c3 = st.columns(3)
        score = textstat.flesch_reading_ease(policy_input)
        grade = textstat.flesch_kincaid_grade(policy_input)
        r_time = textstat.reading_time(policy_input)

        with c1: st.metric("Reading Ease", score)
        with c2: st.metric("Grade Level", f"Grade {grade}")
        with c3: st.metric("Reading Time", f"{round(r_time/60, 2)} min")

        if score > 50: st.success("This text is accessible to the general public.")
        else: st.warning("This text contains complex jargon.")
    st.stop()

# 2. OTP AUTHENTICATION
elif st.session_state.logged_in and not st.session_state.otp_verified:
    st.subheader("🔐 MFA: Email Verification")
    st.write(f"A code has been sent to: **{st.session_state.email}**")

    if st.session_state.generated_otp is None:
        with st.spinner("Sending code..."):
            otp = str(random.randint(100000, 999999))
            if send_otp_email(st.session_state.email, otp):
                st.session_state.generated_otp = otp
                st.success("OTP Sent!")
            else:
                st.error("Email failed. Check SMTP settings.")
                # Fallback for demo if email fails
                st.info(f"DEMO FALLBACK: {otp}")

    otp_in = st.text_input("Enter 6-digit OTP", max_chars=6)
    if st.button("Verify Code"):
        if otp_in == st.session_state.generated_otp:
            st.session_state.otp_verified = True
            st.rerun()
        else:
            st.error("Incorrect code.")
    if st.button("Resend"):
        st.session_state.generated_otp = None
        st.rerun()
    st.stop()

# 3. AUTHENTICATION TABS
else:
    tab1, tab2, tab3 = st.tabs(["Login", "Signup", "Forgot Password"])

    with tab1:
        st.subheader("Login")
        l_email = st.text_input("Email", key="l_email")
        l_pass = st.text_input("Password", type="password", key="l_pass")
        if st.button("Sign In"):
            cursor.execute("SELECT username, password FROM users WHERE email=?", (l_email,))
            user = cursor.fetchone()
            if user and check_data(l_pass, user[1]):
                st.session_state.logged_in = True
                st.session_state.username = user[0]
                st.session_state.email = l_email
                st.rerun()
            else:
                st.error("Invalid credentials.")

    with tab2:
        st.subheader("Create Account")
        s_user = st.text_input("Username", key="s_user")
        s_email = st.text_input("Email", key="s_email")
        s_pass = st.text_input("Password", type="password", key="s_pass")

        if s_pass:
            for rule, met in password_strength(s_pass).items():
                st.write(f"{'✅' if met else '❌'} {rule}")

        s_conf = st.text_input("Confirm Password", type="password", key="s_conf")
        s_q = st.selectbox("Security Question", ["What is your Pet Name?", "What is your Mother's name?", "What is your birthday?"], key="s_q")
        s_a = st.text_input("Answer", key="s_a")

        if st.button("Register"):
            if not all([s_user, s_email, s_pass, s_conf, s_a]):
                st.error("All fields required.")
            elif s_pass != s_conf:
                st.error("Passwords mismatch.")
            elif not valid_password(s_pass):
                st.error("Weak password.")
            else:
                try:
                    cursor.execute("INSERT INTO users(username, email, password, security_question, security_answer) VALUES(?,?,?,?,?)",
                                   (s_user, s_email, hash_data(s_pass), s_q, hash_data(s_a.lower())))
                    conn.commit()
                    st.success("Account created! Login now.")
                except sqlite3.IntegrityError:
                    st.error("Email already exists.")

    with tab3:
        st.subheader("Reset Password")
        if "reset_step" not in st.session_state:
            st.session_state.reset_step = 1

        if st.session_state.reset_step == 1:
            f_email = st.text_input("Email", key="f_email")
            if st.button("Find Account"):
                cursor.execute("SELECT security_question FROM users WHERE email=?", (f_email,))
                res = cursor.fetchone()
                if res:
                    st.session_state.reset_email_locked = f_email
                    st.session_state.reset_q = res[0]
                    st.session_state.reset_step = 2
                    st.rerun()
                else:
                    st.error("Email not found.")

        elif st.session_state.reset_step == 2:
            st.write(f"Question: **{st.session_state.reset_q}**")
            f_ans = st.text_input("Answer", key="f_ans")
            if st.button("Verify Identity"):
                cursor.execute("SELECT security_answer FROM users WHERE email=?", (st.session_state.reset_email_locked,))
                if check_data(f_ans.lower(), cursor.fetchone()[0]):
                    st.session_state.reset_step = 3
                    st.rerun()
                else:
                    st.error("Incorrect answer.")

        elif st.session_state.reset_step == 3:
            new_p = st.text_input("New Password", type="password", key="new_p")
            if st.button("Update Password"):
                if valid_password(new_p):
                    cursor.execute("UPDATE users SET password=? WHERE email=?", (hash_data(new_p), st.session_state.reset_email_locked))
                    conn.commit()
                    st.success("Password changed! Login now.")
                    st.session_state.reset_step = 1
                    time.sleep(1)
                    st.rerun()
                else:
                    st.error("Password is too weak.")

Writing app.py


In [4]:
from pyngrok import ngrok

# Paste your ngrok authtoken below
ngrok.set_auth_token("39c6uE2ms17WzqwFIWSsjZ3Nr9T_39CGdFvPBdecaGdJMSxux")


In [5]:
!streamlit run app.py &>/content/logs.txt &

In [9]:
public_url = ngrok.connect(8501)
public_url


<NgrokTunnel: "https://miyoko-apatetic-elza.ngrok-free.dev" -> "http://localhost:8501">

In [10]:
from pyngrok import ngrok

print("Killing all ngrok tunnels...")
ngrok.kill()
print("All ngrok tunnels killed. You can now try to connect again.")

Killing all ngrok tunnels...
All ngrok tunnels killed. You can now try to connect again.


In [8]:
import os
if os.path.exists("users.db"):
    os.remove("users.db")
    print("Old database deleted. You can now run the app code again.")
else:
    print("Database file not found, you're good to go.")

Database file not found, you're good to go.
